# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define an inverted pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to `Env()` / `Module()` wrappers
3. How `Env` composes modules that share wires

## The inverted pendulum environment

An inverted pendulum balanced upright. The full nonlinear dynamics:

$$\ddot\theta = \frac{g}{\ell}\,\sin\theta + \frac{\tau}{m\ell^2}$$

The positive gravitational term makes the upright equilibrium ($\theta = 0$) **unstable** — without active control, any perturbation grows. The controller must apply torque $\tau$ to keep it balanced.

Write a standard `gym.Env`: `__init__` sets spaces and state, `reset` initializes, `step` updates. Then wrap with `Env()` to extract the symbolic module.

In [1]:
import math
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Env
from zrth.torch import Module


class InvertedPendulumEnv(gym.Env):
    """Inverted pendulum with full nonlinear dynamics."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(2,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.1
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.1
        self.theta_dot = 0.0

        observation = [self.theta, self.theta_dot]
        return observation, {}

    def step(self, torque):
        accel = (self.g / self.l) * math.sin(self.theta) + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = [self.theta, self.theta_dot]
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated, {}

## Wrapping and composition

We create shared wire pairs up front so that the pendulum's observation feeds the controller's input, and the controller's output feeds the pendulum's action. Then we wrap both and compose them into a closed-loop system.

In [2]:
from zrth import Wire, Float

pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)

# Shared wire pairs: pendulum observation (2D) = controller input
obs_wire = (Wire(Float(2)), Wire(Float(2)))
# Shared wire pairs: controller output = pendulum action
act_wire = (Wire(Float(1)), Wire(Float(1)))

wrapped_pendulum = Env(pendulum, observation=obs_wire, action=act_wire)
print(wrapped_pendulum)

module
  external
    w2, w3 : Float(1)
  interface
    w0, w1 : Float(2)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Tensor([0.10000000149011612]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Stack w20; w18, w19
    Id w1; w20
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Div w21; w8, w9
    Sin w22; w4
    Mul w23; w21, w22
    Mul w24; w10, w9
    Mul w25; w24, w9
    Div w26; w2, w25
    Add w27; w23, w26
    Mul w28; w27, w11
    Add w29; w6, w28
    Id w7; w29
    Mul w30; w29, w11
    Add w31; w4, w30
    Id w5; w31
    Stack w32; w31, w29
    Tenso

A small NN that observes $(\theta, \dot\theta)$ and outputs a torque $\tau$. Architecture: $[2] \to 16 \to [1]$. We wrap it with the same shared wires.

In [3]:
class ControllerNN(nn.Module):
    """NN controller: [theta, theta_dot] -> torque. Uses tanh for symmetric output."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, state):
        x = torch.tanh(self.fc1(state))
        return 2.0 * torch.tanh(self.fc2(x))  # scale to [-2, 2]

In [4]:
controller = ControllerNN()
wrapped_controller = Module(controller, extl=obs_wire, intf=act_wire)
print(wrapped_controller)

module
  external
    w0, w1 : Float(2)
  interface
    w2, w3 : Float(1)
  atom controls w3 awaits w1
  init
    Tensor([0.4194094240665436 0.1681482195854187 0.34274405241012573 0.6422670483589172 -0.07558244466781616 ...]) w50; 
    Tensor([-0.33194872736930847 -0.012136269360780716 -0.3053770065307617 -0.10108719021081924 0.6269184350967407 ...]) w51; 
    Linear w52; w1, w50, w51
    Tanh w53; w52
    Tensor([2]) w54; 
    Tensor([-0.1189429759979248 0.20417138934135437 -0.220843106508255 0.05444875359535217 -0.1333574652671814 ...]) w55; 
    Tensor([-0.07621383666992188]) w56; 
    Linear w57; w53, w55, w56
    Tanh w58; w57
    Mul w59; w54, w58
    Id w3; w59
  update
    Tensor([0.4194094240665436 0.1681482195854187 0.34274405241012573 0.6422670483589172 -0.07558244466781616 ...]) w50; 
    Tensor([-0.33194872736930847 -0.012136269360780716 -0.3053770065307617 -0.10108719021081924 0.6269184350967407 ...]) w51; 
    Linear w52; w1, w50, w51
    Tanh w53; w52
    Tensor([2]) w5

In [5]:
closed_loop = Env(wrapped_pendulum, wrapped_controller)
print(closed_loop)

module
  interface
    w0, w1 : Float(2)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
    w2, w3 : Float(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Tensor([0.10000000149011612]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Stack w20; w18, w19
    Id w1; w20
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([9.8100004196167]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([0.05000000074505806]) w11; 
    Div w21; w8, w9
    Sin w22; w4
    Mul w23; w21, w22
    Mul w24; w10, w9
    Mul w25; w24, w9
    Div w26; w2, w25
    Add w27; w23, w26
    Mul w28; w27, w11
    Add w29; w6, w28
    Id w7; w29
    Mul w30; w29, w11
    Add w31; w4, w30
    Id w5; w31
    Stack w32; w31, w29
    Tensor([0]) w33;

In [6]:
from zrth.visual import show
stop = show(closed_loop, names={obs_wire: "θ", act_wire: "τ"})

Module viewer running at http://127.0.0.1:52412 (ws://127.0.0.1:52413)


## Training the controller

We train the controller using **REINFORCE** (policy gradient). The wrapped objects are fully functional — `wrapped_pendulum.reset()` / `wrapped_pendulum.step()` work as gym methods, and `wrapped_controller(obs)` runs a PyTorch forward pass with autograd.

The controller outputs a mean torque $\mu$; we sample from $\mathcal{N}(\mu, \sigma^2)$ to explore. The loss is the standard REINFORCE estimator:

$$\nabla J = \mathbb{E}\left[\sum_t \nabla \log \pi(a_t | s_t) \cdot G_t\right]$$

where $G_t$ is the discounted return from step $t$.

Because `zrth.torch.Module` uses live tensor references, the symbolic module and the composed `closed_loop` automatically reflect the trained weights — no re-wrapping needed.

In [7]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

optimizer = torch.optim.Adam(wrapped_controller.parameters(), lr=0.01)
sigma = 0.3

for episode in range(500):
    wrapped_pendulum.reset()
    log_probs, rewards = [], []

    for step in range(200):
        state = torch.tensor([wrapped_pendulum.theta, wrapped_pendulum.theta_dot], dtype=torch.float32)
        mu = wrapped_controller(state.unsqueeze(0)).squeeze()
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample().clamp(-2.0, 2.0)
        log_probs.append(dist.log_prob(action))

        _, reward, terminated, _, _ = wrapped_pendulum.step(action.item())
        rewards.append(reward)
        if terminated:
            break

    returns = torch.tensor(compute_returns(rewards), dtype=torch.float32)
    loss = -sum(lp * G for lp, G in zip(log_probs, returns))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 100 == 0:
        print(f"episode {episode:3d}  steps = {len(rewards):3d}  reward = {sum(rewards):.2f}")

episode   0  steps =  32  reward = -71.59
episode 100  steps =  28  reward = -75.12
episode 200  steps =  22  reward = -76.86
episode 300  steps =  23  reward = -77.11
episode 400  steps =  23  reward = -71.29


## Evaluating the trained controller

Run the trained controller deterministically (no exploration noise) and print $\theta$ at each step.

In [8]:
wrapped_pendulum.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}  {'torque':>8}  {'reward':>8}")
print("-" * 50)

with torch.no_grad():
    for step in range(20):
        state = torch.tensor([wrapped_pendulum.theta, wrapped_pendulum.theta_dot], dtype=torch.float32)
        torque = wrapped_controller(state.unsqueeze(0)).squeeze().item()
        torque = max(-2.0, min(2.0, torque))

        _, reward, terminated, _, _ = wrapped_pendulum.step(torque)
        print(f"{step:4d}  {wrapped_pendulum.theta:8.4f}  {wrapped_pendulum.theta_dot:10.4f}  {torque:8.4f}  {reward:.4f}")
        if terminated:
            print("terminated!")
            break

step     theta   theta_dot    torque    reward
--------------------------------------------------
   0    0.1074      0.1489    1.9981  -0.0138
   1    0.1225      0.3014    1.9985  -0.0241
   2    0.1456      0.4613    1.9987  -0.0425
   3    0.1772      0.6324    1.9989  -0.0714
   4    0.2181      0.8188    1.9991  -0.1146
   5    0.2694      1.0249    1.9992  -0.1776
   6    0.3322      1.2554    1.9994  -0.2679
   7    0.4079      1.5153    1.9994  -0.3960
   8    0.4984      1.8099    1.9995  -0.5760
   9    0.6056      2.1443    1.9995  -0.8266
  10    0.7318      2.5235    1.9996  -1.1724
  11    0.8794      2.9513    1.9996  -1.6443
  12    1.0508      3.4291    1.9996  -2.2801
  13    1.2486      3.9548    1.9996  -3.1229
  14    1.4746      4.5200    1.9996  -4.2174
  15    1.7300      5.1082    1.9996  -5.6022
  16    2.0146      5.6925    1.9997  -7.2991
  17    2.3264      6.2355    1.9997  -9.3001
  18    2.6610      6.6925    1.9997  -11.5598
  19    3.0120      7.0192 

## Running the composed closed-loop system

The `closed_loop` is an `Env` that composes the pendulum and controller into a single Module, while keeping the real pendulum env for delegation. When we call `step()`, the pendulum atom runs the real env (for rendering, real physics) and the controller atom runs symbolically. Outputs flow between them via shared wires.

Because the symbolic module holds live tensor references to the controller's weights, the composed module already reflects the trained values — no re-wrapping needed.

In [9]:
closed_loop.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}")
print("-" * 30)

for step in range(20):
    obs, reward, terminated, truncated, info = closed_loop.step(0)
    print(f"{step:4d}  {closed_loop.theta:8.4f}  {closed_loop.theta_dot:10.4f}")
    if terminated:
        print("terminated!")
        break

step     theta   theta_dot
------------------------------
   0    0.1024      0.0490
   1    0.1074      0.0991
   2    0.1150      0.1517
   3    0.1254      0.2080
   4    0.1389      0.2693
   5    0.1557      0.3372
   6    0.1764      0.4133
   7    0.2014      0.4994
   8    0.2312      0.5975
   9    0.2667      0.7099
  10    0.3087      0.8391
  11    0.3581      0.9882
  12    0.4161      1.1601
  13    0.4840      1.3583
  14    0.5633      1.5866
  15    0.6558      1.8485
  16    0.7631      2.1476
  17    0.8875      2.4866
  18    1.0308      2.8670
  19    1.1952      3.2877
